# Train on a Colab GPU

Runs `scripts/train.py` on a Colab GPU runtime instead of the local CPU. It clones the repo,
downloads the PhiUSIIL dataset (UCI) and the Tranco list, builds the splits, trains and evaluates.

Connect this notebook to a Colab **GPU** runtime first, either in the VS Code Colab extension or
in the Colab web UI under *Runtime → Change runtime type*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a GPU type"
print(torch.cuda.get_device_name(0))

In [ ]:
REPO = "https://github.com/Kluey/URL-Phishing-Classifier.git"
BRANCH = "hostname-input"

%cd /content
!rm -rf URL-Phishing-Classifier && git clone -q --depth 1 -b {BRANCH} {REPO}
%cd /content/URL-Phishing-Classifier
!git log --oneline -1
!pip install -q -e ".[pretrain]"

In [ ]:
!mkdir -p data model
!wget -q -O /tmp/phiusiil.zip "https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip"
!unzip -o -q /tmp/phiusiil.zip -d data && ls -la data

Builds the hostname splits (PhiUSIIL + Tranco subdomains, all balanced data), then trains a few
configurations and keeps the one with the best **validation** accuracy. The test split is only
used afterwards, for the chosen model.

In [ ]:
import json, shutil

CONFIGS = [(1, 2), (2, 3), (4, 3), (6, 3)]  # (trainable transformer blocks, epochs)
GPT2_DIR = "/content/gpt2"  # outside the repo, so re-cloning keeps the download

results = {}
for i, (blocks, epochs) in enumerate(CONFIGS):
    out = f"model/sweep/blocks{blocks}_epochs{epochs}.pth"
    cmd = (f"python scripts/train.py --trainable-blocks {blocks} --epochs {epochs} "
           f"--output {out} --pretrained-dir {GPT2_DIR}" + (" --prepare-data" if i == 0 else ""))
    !{cmd} 2>&1 | tee -a model/train.log
    run = json.load(open(out.replace(".pth", ".train.json")))
    results[out] = max(run["val_accuracy_per_epoch"])
    print(f"==> {blocks} blocks, {epochs} epochs: best val accuracy {results[out]:.2%}, {run['minutes']:.1f} min")
    print()

best = max(results, key=results.get)
shutil.copy(best, "model/phishing_classifier.pth")
shutil.copy(best.replace(".pth", ".train.json"), "model/phishing_classifier.train.json")
print({k.split("/")[-1]: f"{v:.2%}" for k, v in results.items()})
print("Best by validation accuracy:", best)

In [ ]:
!python scripts/evaluate.py --split test
!python scripts/evaluate.py --split real_world

Files to bring back, each into the same folder in the local project. In VS Code, use the Colab
sidebar → **Contents** → right-click a file → **Download...**.

In [ ]:
!cd /content/URL-Phishing-Classifier && ls -la model/phishing_classifier.pth model/*.json model/train.log data/*.csv data/splits.json | grep -v -e PhiUSIIL -e tranco